<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 4: Batching and Training

#### Tim Moroney, 2026

A lesson where we learn about processing batches of data at once.  Why, how and when, and how it affects the mathematics.

# Package management
We start by installing the required packages. If you're running on Colab this will just download pre-compiled code. Otherwise be prepared to wait a few minutes for installation the first time you run. Feel free to read ahead while you wait.

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# The model setup

One again we set up our CLLM model the same as we've seen previously.

In [ ]:
# Usual set-up
# Load some pre-trained parameters for this model
url = "https://github.com/moroneyt/MXB301/raw/main/resources/CLLM_pretrained.jld2"
paramfile = jldopen(download(url))
p = p0 = paramfile["p"]
close(paramfile)

chars = ['a':'z'; ' ']      # we only deal with lowercase text and space
vocab_size = length(chars)  # number of characters in our "vocabulary"

# Mappings between characters and indices (a = 1, b = 2, etc.)
idx_to_char(i) = chars[i]
char_to_idx(c) = findfirst(isequal(c), chars)
string_to_idxs(s) = [char_to_idx(c) for c in s]

# Forward pass

Here is the `forward` function we derived last time, which returns the cache of intermediate values for use in the later `backward` pass during training.  And the convenience function `model` which does two things:
1. it fixes the parameter input to `p` so we don't have to provide it as an input explicitly.
2. it ignores the returned `cache` of values.

In [ ]:
# Predict probabilities of the next character
function forward(u, p)
    X = p.We[:, u]               # 1. embedding
    v = vec(X)                   # 2. flatten
    z1 = p.W1 * v + p.b1         # 3. first dense layer
    h1 = tanh.(z1)               # 4. activation
    z2 = p.W2 * h1 + p.b2        # 5. second dense layer
    ŷ = softmax(z2)              # 6. softmax for probabilities

    # Return the output along with a cache of intermediate values
    cache = (; u, X, v, z1, h1, z2)
    return ŷ, cache
end

model(u) = forward(u, p)[1]  # holds the parameter values fixed as p

# Test inputs

For this lesson let's build a few different input strings to try with the model.  We can convert each into an array of indices.

In [ ]:
# Four input strings
text = ["quick brow", " lazy foxe", "jumping ov", "should the"]

# Each converted to arrays of indices
idxs = string_to_idxs.(text)

# Batching

For reasons that we will shortly learn about, the usual way that AI models process multiple inputs like this is in the form of a **batch**.  That is, an array with one additional dimension -- the _batch dimension_.  So our input array will be

$$U = [u^{(1)}\ u^{(2)}\ \ldots \  u^{(B)}]$$ where $B$ is the **batch size**.

Each $u^{(j)}$ is the usual vector of indices representing a text sequence of $C$ characters
$$
u^{(j)} = \left[\begin{array}{c}u_1^{(j)}\\ u_2^{(j)}\\ \vdots\\ u_{C}^{(j)}\end{array}\right] \in \{1,\ldots,|\mathcal{V}| \}^{C}
$$

and hence the input $U \in \mathbb{R}^{C \times B}$ is now a matrix with $C = 10$ rows and $B = 4$ columns -- one column for each input string in the batch.

We can build a suitable batched input `U` (upper case) by `stack`ing the representations of each input string as columns of a matrix.

In [ ]:
# The input text as a batch of columns
U = stack(idxs)

# Ground truth
The correct next characters we have in mind are `'n'` ("quick brown"), `'s'` ("lazy foxes"), `'e'` ("jumping ove(r)") and `'y'` ("should they") respectively.  In one-hot encoding, as matrix $Y$ with four columns:

In [ ]:
Y = onehotbatch(char_to_idx.(['n', 's', 'e', 'y']), 1:vocab_size)

# Trying the model on a batch input
Alas, our model cannot currently handle batch inputs.

In [ ]:
model(U)

# Supporting batch inputs

Our first main objective for this lesson is to find out what's going wrong with batch inputs, and to fix the issues.

## The desired output

What we want is for `model(U)` to function as if it was computed over each column of `U`.  That is, as if we had called it one column at a time, and then stacked all the answers together:

$$
\textrm{model}(U) = [\textrm{model}(u^{(1)})\ \ \textrm{model}(u^{(2)})\ \ \cdots\ \textrm{model}(u^{(B)})]\,.
$$

In [ ]:
stack(model(u) for u in eachcol(U))  # this is the output we would like from model(U)

##
Now, we _could_ just rewrite `forward` to calculate the output in exactly this way: one column at a time.  But if we actually take a look at each step of the `forward` code, we will see that it almost works with batches already.  There are actually only a couple of minor changes required to get it working correctly on a batch of inputs.

Let's go through each step of the process to investigate what we need to change.




  The first step

```
X = p.We[:, u]
```

is the embedding, picking out the columns of the embedding matrix $W_e$ that correspond to the input characters.  At the time we wrote the code, we imagined that $u$ would be a vector of indices, and hence $X = W_e[:,u]$ would be the matrix of embeddings, with each column corresponding to a character.

Although we never imagined that $U$ would be a _matrix_ at this step, it turns out to be fine if it is.

The result of $\mathbb{X} = W_e[:,U]$ is a three-dimensional _tensor_, where the third dimension is the batch dimension: $\mathbb{X} \in \mathbb{R}^{d_e \times C \times B}$.  So $\mathbb{X}[:,:,1] \in \mathbb{R}^{d_e \times C}$ is the embedding for the first column of $U$, and $\mathbb{X}[:,:,2]  \in \mathbb{R}^{d_e \times C}$ is the embedding for the second column of $U$, and so on.  So we have exactly what we want here.

In [ ]:
𝕏 = p.We[:, U]  # now for batch inputs

##
The next line in our forward process flattens the embedding into a single long column, ready to input to the dense layer.  And this is where things have gone wrong.

When we wrote the line `v = vec(X)` were imagining that $X$ was a matrix, and this was just flattening, or "vectorising" the matrix into a single long column.

But now that $\mathbb{X}$ contains a batch dimension this isn't correct.  We don't want to dump _all_ of $\mathbb{X}$ into a single huge vector.  Instead we want to vectorise $\mathbb{X}[:,:,1]$ into a column, vectorise $\mathbb{X}[:,:,2]$ into a separate column, and so on.  In other words, $V$ needs to be a matrix, with each column corresponding to a vectorised matrix from the batch.


##
So instead of the (incorrect) code which produces a single huge column

In [ ]:
V_wrong = vec(𝕏)  # this is not what we want

##
we instead want to reshape the result into a matrix.  Since the batch size is $B = $`size(U,2)`, the correct code to do the flattening using `reshape` is shown here.

Note that we don't need to specify the number of rows in the call to `reshape` -- we can use use a `:` for that.  Since we are reshaping, the total number of entries can't change, so one dimension of the new shape can be inferred.  Here because $\mathbb{X}$ is a $2 \times 10 \times 4$ array,  reshaping it into a matrix with $4$ columns implies it must have $2 \times 10 = 20$ rows.

In [ ]:
V = reshape(𝕏, :, size(U,2))

## Dense layer

Now each column of $V$ is the flattened embedding for each input from the batch.  Will the dense layer that follows now function correctly?  Its formula is

`z1 = p.W1 * v + p.b1`

Let's try the matrix multiply with the weights first.

And actually it's fine.  Previously this was a matrix-vector multiply, and now it's a matrix-matrix multiply.  Having the batch dimension as the columns of $V$ makes this step "just work" without any changes.  Very nice.

In [ ]:
p.W1 * V

## Bias term

The addition of the bias term needs a change though, albeit a very minor one.  Here it is first not working.

In [ ]:
Z1_wrong = p.W1 * V + p.b1

##
The problem is that the bias is a vector, and we can't add a matrix and a vector.  What we really want is for the bias vector to be added to _each column_ of the matrix.  So that's just a broadcast using `.+` rather than `+`.

That's the only change required to the dense layer.

In [ ]:
Z1 = p.W1 * V .+ p.b1   # dot plus here

## Activation function
The next line of the forward pass is just the element-wise activation function.  Since it's already broadcast with `.` nothing needs to change here.

In [ ]:
H1 = tanh.(Z1)

## Next dense layer

Then comes another dense layer.  Again, we're just going to need the "dot plus" to make this work.

In [ ]:
Z2 = p.W2 * H1 .+ p.b2   # working fine with broadcast vector addition

## Softmax

And finally `softmax`, which by default applies over each column, to support exactly the situation we have here.  So no change required.

In [ ]:
Ŷ = softmax(Z2)

## Batch-aware model code

So with these few minor modifications, our `forward` pass now fully supports a batch dimension.

In [ ]:
# Predict probabilities of the next character
function forward(U, p)
    𝕏 = p.We[:, U]                 # 1. embedding
    V = reshape(𝕏, :, size(U,2))   # 2. flatten
    Z1 = p.W1 * V .+ p.b1          # 3. first dense layer
    H1 = tanh.(Z1)                 # 4. activation
    Z2 = p.W2 * H1 .+ p.b2         # 5. second dense layer
    Ŷ = softmax(Z2)                # 6. softmax for probabilities

    # Return the output along with a cache of intermediate values
    cache = (; U, 𝕏, V, Z1, H1, Z2)
    return Ŷ, cache
end

## Test on our input data

We can call it now (via our `model` helper) to examine the model's predicted probabilities for the next character of each text sequence "quick brow", " lazy foxe", "jumping ov", "should the".

In [ ]:
Ŷ = model(U)

##
We could use deterministic sampling to continue the text sequences by finding the `argmax` for each column.

In [ ]:
nextchar = idx_to_char.(argmax.(eachcol(Ŷ)))

##
The model's most probable predicted continuations are shown below.  Apart from `"jumping ove"` they don't match the ground truth.  But each is a legitimate-looking continuation all the same.

In [ ]:
text .* nextchar

# Loss and gradient

Now that we have the forward pass working, we can see about calculating the loss.  Importantly, the loss function is a _sum_ over the training examples $(u^{(j)}, y^{(j)})$ (remember we are using superscripts to indicate different training examples):

$$
L(p) := \sum_{j=1}^{B} \ell(y^{(j)}, \hat{y}^{(j)}) = \sum_{j=1}^{B} \ell(y^{(j)}, M_p(u^{(j)}))
$$

where $\ell$ is the **per-sample loss function** -- for our example, $\ell = \textrm{crossentropy}$.  So the gradient is a sum too:

$$
\nabla_p L = \sum_{j=1}^{B} \nabla_p \ell(y^{(j)}, M_p(u^{(j)}))\,.
$$

This simplifies matters considerably.

The `crossentropy` library function already knows to sum over training examples, so the loss _value_ works out of the box without any modification.

In [ ]:
crossentropy(Y, Ŷ)

# Backward pass
So the loss value is fine as it is.  What about the gradients?  Let's work through the backward pass that we derived last time to check.

In [ ]:
# Backward pass: compute gradients
function backward(p, y, ŷ; cache)

    # gradient vector to fill in
    g = zero(p)

    # unpack the cache of intermediate values from the forward pass
    (; u, X, v, z1, h1, z2) = cache

    # Calculate the gradient of the loss with respect to the model parameters.
    # Each rule is simple enough, but take care!
    ∇z2 = ŷ - y                         # 6. softmax with crossentropy loss
    g.W2 = ∇z2 * h1'                    # 5. second dense layer (wrt weights)
    g.b2 = ∇z2                          # 5. second dense layer (wrt bias)
    ∇h1 = p.W2' * ∇z2                   # 5. second dense layer (wrt data)
    ∇z1 = (1 .- h1.^2) .* ∇h1           # 4. activation
    g.W1 = ∇z1 * v'                     # 3. first dense layer (wrt weights)
    g.b1 = ∇z1                          # 3. first dense layer (wrt bias)
    ∇v = p.W1' * ∇z1                    # 3. first dense layer (wrt data)
    ∇X = reshape(∇v, size(X))           # 2. (un)flatten
    g.We = scattergrad(+, ∇X, u)        # 1. embedding (wrt weights)

    return g
end

# Softmax with cross-entropy
The softmax cross-entropy combination has a simple gradient, which continues to work fine with batches.  As usual each column here is from one training example in the batch.

In [ ]:
∇Z2 = Ŷ - Y

# Dense layer

Now it's time to compute $\nabla_{W_2} L$ and $\nabla_{b_2} L$ and we need to ensure the contributions from each training example are _summed_.

Our formula from lesson 3, the classic dense layer backprop result, was

$$
\nabla_{W_2} L = \nabla_{z_2} L\ {h_1}^\top
$$

where $\nabla_{z_2} L$ and ${h_1}^\top$ were both vectors.  In fact _this formula still holds_ even though both $\nabla_{Z_2} L$ and ${H_1}^\top$ are now matrices.  Let's see why.

Write the two matrices out in columns:

$$
\nabla_{Z_2} L = \left[(\nabla_{Z_2} L)_{(:,1)}\ \ (\nabla_{Z_2} L)_{(:,2)} \  \ \cdots \  (\nabla_{Z_2} L)_{(:,B)}\right]
$$

and

$$
H_1 = \left[(H_1)_{(:,1)}\ \ (H_1)_{(:,2)} \  \ \cdots \  (H_1)_{(:,B)}\right]\,.
$$

##
Then
$$
\begin{align*}
\nabla_{Z_2} L\ {H_1}^\top &= \left[(\nabla_{Z_2} L)_{(:,1)}\ \ (\nabla_{Z_2} L)_{(:,2)} \  \ \cdots \  (\nabla_{Z_2} L)_{(:,B)}\right] \left[\begin{array}{c} {(H_1)_{(:,1)}}^\top \\ {(H_1)_{(:,2)}}^\top \\ \vdots \\ {(H_1)_{(:,B)}}^\top \end{array}\right]\\
&= \sum_{j=1}^{B} (\nabla_{Z_2} L)_{(:,j)}\, {(H_1)_{(:,j)}}^\top
\end{align*}
$$

which is exactly the sum of the contributions from each training example.  So once again the linear algebra is doing the right thing for us automatically.

In [ ]:
∇W2 = ∇Z2 * H1'   # still the correct formula for ∇W2

##
For the bias gradient $\nabla_{b_2} L$ we will need to sum the columns ourselves though:

$$
\nabla_{b_2} L = \sum_{j=1}^{B} (\nabla_{Z_2} L)_{(:,j)}
$$

In [ ]:
∇b2 = sum(∇Z2; dims=2)  # now with a sum over the columns

## Corrected backward pass code
Making these changes throughout, we are done with the backward pass.  The final `scattergrad` operation, being an NNlib library function, handles the batch dimension properly already, provided you reshape the inputs into the sizes it expects.

In [ ]:
# Backward pass: updates gradient in-place
function backward(p, Y, Ŷ; cache)

    # gradient vector to fill in
    g = zero(p)

    # unpack the cache of intermediate values from the forward pass
    (; U, 𝕏, V, Z1, H1, Z2) = cache

    # Calculate the gradient of the loss with respect to the model parameters.
    # Each rule is simple enough, but take care!
    ∇Z2 = Ŷ - Y                         # 6. softmax with crossentropy loss
    g.W2 = ∇Z2 * H1'                    # 5. second dense layer (weights)
    g.b2 = sum(∇Z2; dims=2)             # 5. second dense layer (bias)
    ∇H1 = p.W2' * ∇Z2                   # 5. second dense layer (data)
    ∇Z1 = (1 .- H1.^2) .* ∇H1           # 4. activation
    g.W1 = ∇Z1 * V'                     # 3. first dense layer (weights)
    g.b1 = sum(∇Z1; dims=2)             # 3. first dense layer (bias)
    ∇V = p.W1' * ∇Z1                    # 3. first dense layer (data)
    ∇𝕏 = reshape(∇V, size(𝕏, 1), :)     # 2. (un)flatten
    g.We = scattergrad(+, ∇𝕏, vec(U))   # 1. embedding (weights)

    return g
end

# Loss and gradient

We haven't defined the loss function yet.  For training the model, it's going to be just as important to know the gradient of the loss as it is to know its value. Computing the loss value requires the `forward` pass through the model, and computing the gradient requires a subsequent `backward` pass through the model.

In which case, we may as well bundle both steps up into a single function, call it `loss_and_gradient`. We'll add in a `grad=true` default option, but the user can pass `grad=false` to compute the loss only (forward pass only) and no gradient.

For the loss, we'll use the _mean_ of the cross entropy rather than the sum.  It's just a division by a constant, so makes no real difference to anything, but it ensures the loss value being printed is not sensitive to the number of training examples you happen to be using.

In [ ]:
function loss_and_gradient(p; U, Y, grad=true)

    B = size(U, 2) # batch size
    @assert B == size(Y, 2)

    # Forward
    Ŷ, cache = forward(U, p)
    loss = crossentropy(Y, Ŷ) / B
    if !grad return loss end

    # Backward
    g = backward(p, Y, Ŷ; cache) / B

    return loss, g

end

## Checking gradients
It's always wise to check the gradients using an autodiff tool before proceeding.   We want reverse mode AD, so we'll use `Zygote`.

From the output below we see the gradients agree precisely.

Excellent, we have the forward and backward pass functioning correctly with a batch dimension now.

In [ ]:
# Compute the gradient using our code and with reverse mode AD

# Our code
loss, g = loss_and_gradient(p; U, Y)

# Reverse mode AD
g_AD = gradient(p -> loss_and_gradient(p; U, Y, grad=false), AutoZygote(), p)

# Do they agree?
maximum(abs, g - g_AD)

# Why batching?

We mentioned earlier that deep learning libraries default to processing inputs in batches, rather than one sample at a time.  Why is this?  The answer is efficiency.  This section closely follows the presentation in [All About Rooflines](https://jax-ml.github.io/scaling-book/roofline/).

A note on terminology: we will use FLOP as shorthand for "floating point operation(s)".  It can be singular or plural.  We will use FLOP/s as shorthand for "floating point operations per second".  To avoid confusion we will _not_ speak of FLOPs (plural).  So if there's an s, it's always /s, meaning per second.

Typically in deep learning models, matrix multiplications dominate the workload.   We hardly notice the runtime for our toy example, but real AI models are huge and run on very fast, specialised hardware, either GPUs (graphics processing units) or TPUs (tensor processing units) which are highly optimised for matrix multiplications.

To actually calculate a matrix multiplication on a GPU (say) requires both **communication** and **computation**.  Communication here refers to the act of loading the matrix operands from device memory, and also writing the result to device memory.  Computation refers to performing the required FLOP to actually multiply the operands to produce the result.  Communication and computation both take time.

For communication, the time required can be expressed as
$$
T_\textrm{mem} = \frac{\textrm{Communication bytes}}{\textrm{Bandwidth [bytes / s]}}
$$
The bandwidth is the speed at which the device and read and write from memory. For example, a GPU may have a  bandwidth of $500$ GB/s = $500 \times 10^9$ bytes / second.

For computation, the time required can be expressed as
$$
T_\textrm{comp} = \frac{\textrm{Computation FLOP}}{\textrm{Compute speed [FLOP/s]}}
$$
For example, a GPU may have a compute speed of $10$ TFLOP/s = $10^{12}$ FLOP/s.

If $T_\textrm{comp} > T_\textrm{mem}$ then the workload is said to be **compute-bound**. That's good, it means the GPU is fully utilised.  While its compute cores are busy performing the actual computations, the memory unit of the device can shuffle the required data in and out of memory ready for the next round.  So the communication overhead is essentially hidden.

Conversely, if $T_\textrm{mem} > T_\textrm{comp}$ then the workload is said to be **communication-bound**.  That's bad. It means the GPU cores are spending some of their time waiting around with nothing to do, while the data they need to compute with is being shuffled around.

So to make full use of the very expensive GPU hardware, it's essential that $T_\textrm{comp} > T_\textrm{mem}$.

Let's analyse this inequality $T_\textrm{comp} > T_\textrm{mem}$ in more detail.

$$
\frac{\textrm{Computation FLOP}}{\textrm{Compute speed [FLOP/s]}} >
\frac{\textrm{Communication bytes}}{\textrm{Bandwidth [bytes / s]}}
$$

Multiply and divide through to express this in the following equivalent form:

$$
\begin{align*}
\frac{\textrm{Computation FLOP}}{\textrm{Communication bytes}} &>
\frac{\textrm{Compute speed [FLOP/s]}}{\textrm{Bandwidth [bytes / s]}}\qquad\qquad(*)
\end{align*}
$$

The quantity on the right of inequality $(*)$, called the **peak operating intensity**, can be calculated for a particular device.  For the example values we quoted earlier we have
$$
\begin{align*}
\textrm{peak operating intensity} &= \frac{\textrm{Compute speed [FLOP/s]}}{\textrm{Bandwidth [bytes / s]}}\\
&= \frac{10^{12} \textrm{ FLOP/s}}{500 \times 10^9 \textrm{ bytes / second}}\\
&= 20 \textrm{ FLOP / byte}\,.
\end{align*}
$$

Hence we should try to arrange our workload so that the quantity on the left of $(*)$ is at least as high as this peak operating intensity, to ensure full utilisation of the GPU.

Let's now analyse this quantity on the left for the case of matrix multiplication
$$
Z = W V
$$
where $W$ is dimension $m \times n$ (usual weight matrix) and $V$ is $n \times B$, where $B$ is the batch size.  If we assume each entry is a 32-bit floating point number, then loading $W$ and $V$ from memory requires transferring $4mn + 4nB$ bytes.  Writing the result $Z$ to memory requires transferring $4mB$ bytes.

Meanwhile the computation required is
$$
Z_{ij} = \sum_{k=1}^n W_{ik} V_{kj}, \quad i=1,\ldots,m,\ j=1,\ldots,B
$$
for $2mnB$ FLOP in total -- three nested loops with one multiplication and one addition per iteration.

The left hand side of inequality $(*)$ then, the **intensity** of matrix multiplication is

$$
\begin{align*}
\textrm{intensity (matmul)} &= \frac{\textrm{Computation FLOP}}{\textrm{Communication bytes}}\\ &= \frac{2mnB \textrm{ FLOP}}{(4mn + 4nB + 4mB) \textrm{ byte} }\\
&= 0.5 \frac{mnB}{mn + nB + mB} \textrm{ FLOP / byte}\,.
\end{align*}
$$

$$
\require{cancel}
$$

If we assume the weight matrix is large compared to the batch size, i.e. $m \gg B$ and $n \gg B$, we can simplify the expression above

$$
\textrm{intensity (matmul)} \approx 0.5 \frac{mnB}{mn + \xcancel{nB} + \xcancel{mB}} = 0.5 \frac{mnB}{mn} = 0.5 B \quad \textrm{FLOP / byte.}
$$

Hence we see that processing one sample at a time (batch size $B = 1$) we achieve only $0.5$ FLOP / byte, which is nowhere close to hitting the peak operating intensity of $20$ FLOP / byte.  Only by choosing a sufficiently large batch size (here it would be $B = 40$ if all the calculations were taken literally) can we fully utilise the GPU.

Now, in practice the true story is far more complex, depending on other factors like cache, tiling, and other technicalities of GPUs.  But the key qualitative takeaway is correct: _batching is essential to achieve peak performance._

## Gradient descent

OK, back to our model now.  We have successfully modified it to support batching in both the forward and backward pass.  We can compute gradients.  We have a total of (only!) 4 training examples.  That's enough to have a first little try at training the model.  In the next lesson we will take training much more seriously.

The first method for training we will consider is **gradient descent**. The basic principle of the method is that the gradient (of anything) points in the direction of the steepest local increase.  We can see this by returning to the defintion (assuming infinitesimal variations)

$$
\delta L = \langle \nabla_p L, \delta p \rangle\,.
$$

The watertight way to proceed is using the [Cauchy-Schwarz](https://en.wikipedia.org/wiki/Cauchy%E2%80%93Schwarz_inequality) inequality.  But the right intuition can be derived by recalling the angle between vectors in an inner product space:

$$
\langle u,v \rangle = \|u\| \|v\| \cos\theta
$$

where $\theta$ is the angle between $u$ and $v$.  So returning to $\delta L$,

$$
\delta L = \langle \nabla_p L, \delta p \rangle = \|\nabla_p L\| \|\delta p\| \cos\theta\,.
$$

So the inner product, and hence $\delta L$, is maximised when $\theta = 0$, i.e. when $\delta p$ is in the direction of $\nabla_p L$.  And more to the point for our purposes, it is minimised when $\theta = \pi$, i.e. when $\delta p$ is in the direction _opposite to_ $\nabla_p L$.

For notational convenience let
$$
g = \nabla_p L\,.
$$

Then the direction of $-g$ is a **descent direction**, at least instantaneously, meaning an infinitesimal perturbation of $p$ in this direction will decrease $L$.  For some sufficiently small $\alpha$, called the **learning rate**,

$$
L(p - \alpha g) < L(p)\,.
$$

That one line of maths is enough to propose a crude parameter optimisation scheme, based on the update
$$
p \leftarrow p - \alpha g\,.
$$


```
choose α "sufficiently small"
repeat
  g = ∇ₚL(p)
  p ⬅ p - α g
end
```

This scheme is known as **gradient descent**.

## Gradient descent example
Let's give it a try.  We'll just pick $\alpha = 0.1$ rather arbitrarily, and take 20 iterations.

After 20 steps of gradient descent the loss is reduced to about 5% of its initial value.

In [ ]:
p = p0               # start the iteration from the initial pre-trained parameters
gd_loss_trace = []   # we will record the loss values for each step

α = 0.1  # learning rate

# Gradient descent iterations
for i = 1:20
    loss, g = loss_and_gradient(p; U, Y)
    @show loss # print the current loss value
    push!(gd_loss_trace, loss)  # record the loss value

    p -= α * g   # take the gradient descent step
end

##
This is more than sufficient for the model to now correctly complete the four text sequences, assuming we use deterministic (`argmax`) sampling.

In [ ]:
text .* idx_to_char.(argmax.(eachcol(model(U))))

# Newton's method

The main weakness of gradient descent is that it uses only first-order derivative information about the loss function: its gradient.  The descent direction it chooses is therefore only locally optimal (in the sense of an infinitesimally small learning rate $\alpha$), and it may be that much better search directions would be available, but are not apparent without making use of higher-order derivative information.

**Newton's method** is a very famous minimisation algorithm (it also has a variant for solving nonlinear equations, which you may be familiar with) which makes full use of second-order _curvature_ information in the loss function to choose a search direction.

Suppose $L(p)$ has the Taylor series expansion
$$
L(p+\Delta p) = L(p) + \nabla_p L \cdot \Delta p + \frac{1}{2} \Delta p^\top H \Delta p + \mathcal{o}(\|\Delta p\|^2)
$$

where $H$ is the _Hessian matrix_ of mixed second order partial derivatives
$$
H_{ij} = \frac{\partial^2 L}{\partial p_i\, \partial p_j}\,.
$$

Ignoring the higher order terms, the right hand side of the above expansion is a local quadratic model $m$ approximating the true loss function:

$$
L(p+\Delta p) \approx m(\Delta p)
$$

where $m$ is defined by (using the variable $s$ for convenience)

$$
m(s) = a + g^\top s + \frac{1}{2} s^\top H s
$$

where $a = L(p)$ is the loss value, and $g = \nabla_p L$ is the gradient.


We want $L(p + \Delta p)$ to be minimal.  For the true loss function $L$ this is an intractable problem to solve exactly.  But for the quadratic model $m$ we can readily find the exact minimum.  Since we want $m(s)$ minimised we solve for the stationary point $\nabla_s m = 0$.

Using our matrix calculus expertise, we find the gradient of $m$ is (exercises!):

$$
\nabla_s m = g + H s\,.
$$

So to find the stationary point $s^*$ where $\nabla_s m = 0$, we solve

$$
g + Hs^* = 0 \implies H s^* = -g
$$

and hence choose the Newton step
$$
\Delta p = s^* = -H^{-1} g\,.
$$

Notice that this is more than just a search direction -- it's a full step, intended to be used unscaled:

$$
p \leftarrow p + \Delta p,\qquad \textrm{where} \qquad H \Delta p = -g\,.
$$

This is one step of Newton's method.  There are several points to note.  As already mentioned, Newton's method is making use of second-order, i.e. curvature, information, in the form of the Hessian matrix.  That is how it is able to make a definite recommendation about the size of the update to $p$, and not just propose a descent direction, as with gradient descent.

Another point is that this looks to be tremendously expensive!  The Hessian is a full $P \times P$ matrix, where $p \in \mathbb{R}^P$.  If $P$ is in the thousands, let alone millions or higher, it can be quite impractical to compute and store this matrix which is $\mathcal{O}(P^2)$ entries.  And worse still, the linear solve costs $\mathcal{O}(P^3)$ operations, which  is completely out of the question for large $P$.  We will have more to say on these points in the next lesson.

A third point is that the update proposed by Newton's method is based on the assumption that the local quadratic model $m$ is an accurate representation of the true loss function $L$.  And it's true that for a sufficiently smooth function $L$, the Taylor expansion is accurate in a neighbourhood of the point $p$.  But this neighbourhood might be small, and might not be anywhere close to encompassing the actual minimum of $L$.  In fact, there is no guarantee that the Hessian matrix $H$ is positive definite in general -- the solution to $H \Delta p = -g$ could well be a saddle point.

It is really only when the current iterate $p$ is close to a true minimum of $L$ that we can fully trust the minimiser of the local quadratic model $m$.  And so Newton's method cannot really be used reliably in practice without further modifications.

## Regularisation

One method of _regularisation_ for Newton is to replace the Hessian linear system with the regularised version

$$
(H + \gamma I) \Delta p = -g
$$

where the addition of the positive diagonal term $\gamma I$ ensures, for sufficiently large $\gamma$, that the resulting matrix is positive definite.

But how to choose $\gamma$? When $p$ is close to the true minimiser, you want $\gamma \to 0$ to recover the true Newton's method, so one idea is just to set

$$
\gamma = c\,\| g \|
$$

for some positive constant $c$.  In that case the "regularised Newton" update looks like

$$
(H + c\| g \| I) \Delta p = -g\,.
$$

When $\| g \|$ is large (so you're a long way from converged) the second term in the sum dominates, so this looks more like a gradient descent update

$$
\Delta p \approx \frac{-1}{c \| g \|} g
$$

with learning rate $\alpha =  \frac{1}{c \| g\|}$.

Whereas when $\| g \|$ is small (so you're close to converged), you recover the full Newton update

$$
\Delta p \approx -H^{-1} g\,.
$$

##
So let's give regularised Newton a try on our problem. How do we obtain the Hessian?  We absolutely will not be deriving analytic formulas for the _second derivatives_ by hand!  That's just too much matrix calculus for our liking.

However, because we have an analytic _gradient_, we can use automatic differentiation to go from there to get the Hessian. The Hessian is just the Jacobian of the gradient, after all. That is, if

$$
G_i = \frac{\partial L}{\partial p_i}\qquad\textrm{then}\qquad
H_{ij} = \frac{\partial G_i}{\partial p_j}\,.
$$

This is not going to be cheap to compute.  No matter how you look at it, you are differentiating $P$ equations with respect to $P$ unknowns, so it's $\mathcal{O}(P^2)$ work.  Given that, we opt for the simplest method: forward mode AD.  There is no point paying the extra price for the additional bookkeeping of reverse mode AD if it's $\mathcal{O}(P^2)$ work either way.

In [ ]:
H(p) = jacobian(p->loss_and_gradient(p; U, Y)[2], AutoForwardDiff(), p)
H(p)

##
But we clearly see the expense of forming the Hessian versus the gradient.

In [ ]:
@time loss_and_gradient(p; U, Y);
@time H(p);

##
Still, maybe the expense is worth it?  Here's Newton's method with regularisation applied to our model problem, using $\gamma = \|g\|$ in the regularisation term.  We reset the parameters to their initial values first, and then run for the same number of iterations as we did for gradient descent.

In [ ]:
# Newton's method with regularisation
p = p0
newton_loss_trace = []

for i = 1:20
    loss, g = loss_and_gradient(p; U, Y)
    @show loss # print the current loss value
    push!(newton_loss_trace, loss)

    γ = norm(g)         # regularisation factor
    A = H(p) + γ*I      # regularised Hessian
    Δp = A \ -g         # Newton step
    p += Δp             # take the step
end

##
The improvement in the convergence rate of Newton in comparison to gradient descent is quite stark.  In the same number of iterations, regularised Newton has reduced the loss by more than 5 orders of magnitude.

We can see this even more clearly if we plot the normalised loss (i.e. loss / initial loss) against iterations, on a semilog scale.

In [ ]:
fig, ax = lines(gd_loss_trace/gd_loss_trace[1], label="Gradient descent (fixed α)",
                axis=(title = "Training progress", xlabel = "iterations", ylabel = "normalised loss", yscale = log10))
lines!(newton_loss_trace/newton_loss_trace[1], label="Regularised Newton")
axislegend(ax)
fig

# Conclusion

In this lesson we learned:

* how to extend our character-level language model from single examples to a batch
* how the formulas for the forward and backward pass change (if at all) for batches
* why batching is essential for achieving good performance on modern hardware
* the basic idea of gradient descent
* the basic idea of Newton's method, and how to regularise it
* the implications about computational expense and computational efficiency of the two methods

In the next lesson we will build on this to develop some more practical optimisation methods, and finally train our model with real data.